# HOUSE-3 · 5 — The book

**The failure this notebook prevents.** At the end of the year the house has four
search readouts from three advertisers and pools them into "search drives about seven
percent of conversions" with a tight interval. Three things are wrong with that
sentence before any arithmetic. Two of the four are acme's — one client, one team, one
seasonality — and were counted as two independent draws. One of them was read on the
markets whose feed stayed up, which is a different population. And "conversions" meant
a trial start to acme until March and a paid subscription after, and nothing in a
`StudyRecord` says which.

Then the house takes the pooled number back to northwind and uses it to fix the model
that started this: the media-mix model whose contribution was never the lift.

| | reaches into |
|---|---|
| the corpus | `meta.StudyRecord`, `meta.Corpus`, `meta.record_from_summary` |
| what the pool would be averaging | `meta.commensurable`, `meta.Commensurability`, `estimands.Estimand.transfer_to`, `core.LatentSelection`, `io.DefinitionRegistry` |
| the party level | `meta.pool`, `meta.PoolSpec`, `meta.PoolPriors`, `meta.three_level`, `meta.variance_shares`, `meta.prior_from_pool` |
| across a scope boundary | `io.Catalog.transfer`, `io.Catalog.crossings`, `core.LedgerLine` |
| the model, calibrated | `calibrate.agreement`, `calibrate.fit_calibrated`, `calibrate.derive_prior`, `surface.response_band`, `adapters.contribution` |

In [ ]:
import sys

sys.path.insert(0, ".")

import numpy as np
import pandas as pd
from IPython.display import display

import house as H
import year as Y
from axiom.display import enable, table
from _style import (
    AQUA, BLUE, CRITICAL, GOOD, MUTED, ORANGE, VIOLET, WARNING,
    annotate, caption, compare, curve_band, density, dumbbell, figure, heat, intervals,
    lines, mark_x, mark_y, points, shade, steps,
)

enable();  # every axiom result renders itself from here on

worlds = {name: H.world(name, year=1) for name in H.PARTIES}
last_year = {name: H.world(name, year=0) for name in H.PARTIES}
inputs = {name: Y.design_inputs(w) for name, w in last_year.items()}
catalog, programs, builders = Y.open_house()
Y.register_definitions(builders)
plans = Y.plan_year(builders, inputs)
fielded = Y.field_year(worlds, plans)
reads = Y.read_year(worlds, builders, plans, fielded)
definitions = builders["northwind"].definitions
print({k: r.run.conformance().status for k, r in reads.items()})

## 1 · The corpus

A `StudyRecord` pools a *dimensionless* quantity. A lift in conversions per market-week
is not one — northwind's conversions are orders and globex's are cases — so each search
readout enters as a `relative_change`: the lift as a share of the control arm's
conversions in the window. Four licensed search readouts from three parties, plus the
complier effect GX-03 carried on its ledger, filed as a record of its own so that the
pool can be shown refusing it.

In [ ]:
from axiom.core import LatentSelection, Population
from axiom.identify import compliance
from axiom.meta import Corpus, StudyRecord, record_from_summary

SEARCH = [k for k, r in reads.items() if r.plan.experiment.channel == "search" and r.run.conformance().licensed]

def relative(key):
    rd = reads[key]
    f = rd.fielded
    kpi = Y.kpi_seen(worlds[f.experiment.party], f)
    base = float(np.nanmean(kpi[f.control_indices()][:, f.experiment.post]))
    return rd.measurement.estimate / base, rd.measurement.se / base, base

records, estimands = [], {}
for key in SEARCH:
    rd = reads[key]
    est, se, base = relative(key)
    population = Population(name="markets_whose_feed_stayed_up") if key == "AC-04" else None
    estimands[key] = Y.lift_estimand(rd.plan.experiment, population=population)
    records.append(record_from_summary(study=key, contributor=rd.plan.experiment.party, quantity="relative_change",
                                       estimate=est, se=se, read="experiment", family="search", n=H.N_MARKETS,
                                       estimand_hash=estimands[key].content_hash(),
                                       period=f"weeks {rd.plan.experiment.start}-{rd.plan.experiment.stop}", source=key))

# GX-03's complier effect, from the same frame notebook 3 built
rd = reads["GX-03"]; f = rd.fielded; exp = f.experiment
kpi = Y.kpi_seen(worlds["globex"], f)
change = kpi[:, exp.post].mean(axis=1) - kpi[:, exp.pre].mean(axis=1)
frame = pd.DataFrame({"change": change, "assigned": [1.0 if m_ in f.treated else 0.0 for m_ in H.MARKETS],
                      "off": [1.0 if (m_ in f.treated and f.delivered_fraction[m_] > 0) else 0.0 for m_ in H.MARKETS]})
comp = compliance(frame, "change", "assigned", "off")
_, _, base = relative("GX-03")
compliers = LatentSelection(kind="complier", instrument="assigned", exposure="off", share=comp.share)
estimands["GX-03-compliers"] = Y.lift_estimand(exp, population=Population(name="all_markets", latent=compliers))
records.append(record_from_summary(study="GX-03-compliers", contributor="globex", quantity="relative_change",
                                   estimate=-comp.complier.estimate / base, se=comp.complier.se / base, read="experiment",
                                   family="search", n=H.N_MARKETS, estimand_hash=estimands["GX-03-compliers"].content_hash(), source="GX-03"))
corpus = Corpus(records=tuple(records), name="search, 2026")
table([[r.study, r.contributor, f"{r.estimate:+.3f}", f"{r.se:.3f}", r.period or "—"] for r in corpus.records],
      headers=("record", "party", "relative lift", "se", "period"), title=f"{len(corpus)} records, {len(corpus.contributors())} parties")

## 2 · What would the pool be averaging?

`commensurable` takes the `Estimand` behind each record and transfers it to a
reference. Every facet that differs is either an assumption the pool would be making
silently — and is now on the result — or a reason the pool refuses. The complier
record is the second kind: a precision-weighted average over the compliers of one
experiment and the whole population of another is a number over no population at all.

In [ ]:
from axiom.meta import Commensurability, commensurable

agreed: Commensurability = commensurable(corpus, estimands, reference="NW-01")
print(agreed.ledger_line().statement)
print("verdict:", agreed.verdict().status, "| poolable:", agreed.poolable)
table([[e.study, e.facet, e.status, e.reason[:110]] for e in agreed.entries],
      headers=("record", "facet", "the transfer said", "why the pool refuses"))
print("\nassumptions the pool would be making about the rest:", [a.name for a in agreed.assumptions])
plan = estimands["AC-04"].transfer_to(estimands["NW-01"])
print("AC-04 -> NW-01:", plan.status, "| differing facets:", plan.differing)

Two parties measuring different *populations* is a downgrade, not a block — that
heterogeneity is what a random-effects pool is for, and the assumption rides on the
result. The complier record comes out, and the four that remain are commensurable
under three named assumptions.

### What did each party mean by a conversion?

The registry adds the check one level further back. Each record's party's *current*
definition of `conversion` is resolved through the catalog, and a party that means
something different from the reference's is an incompatibility on the outcome facet.
Today the three agree. Had the house pooled in March — acme's two records did not
exist yet, but the check does not know that — it would have refused.

In [ ]:
poolable = Corpus(records=tuple(r for r in corpus.records if r.study != "GX-03-compliers"), name="search, 2026")
pool_estimands = {k: v for k, v in estimands.items() if k != "GX-03-compliers"}

now = commensurable(poolable, pool_estimands, reference="NW-01", definitions=definitions, programs=programs, term="conversion")
print("with today's definitions:", now.verdict().status, "| entries:", [(e.study, e.facet) for e in now.entries] or "none")

march_catalog, march_programs, march_builders = Y.open_house()
Y.register_definitions(march_builders, through_week=10)
march = commensurable(poolable, pool_estimands, reference="NW-01",
                      definitions=march_builders["northwind"].definitions, programs=march_programs, term="conversion")
print("with the definitions as they stood in March:", march.verdict().status)
table([[e.study, e.facet, e.reason[:120]] for e in march.entries], headers=("record", "facet", "why"))
for d in definitions.history(programs["acme"], "conversion"):
    print(f"acme v{d.version}, registered {d.registered}: unit {definitions.get(programs['acme'], 'conversion', d.version).unit!r}"
          + (f" — {d.note}" if d.note else ""))

## 3 · Four records from three parties are not four draws

`effect_key="study"` gives every record its own effect and credits the family mean
with four independent units. `effect_key="nested"` puts a party level above the
studies: acme's two search tests share a party effect, and the mean's uncertainty is
scaled to the number of parties. The Laplace backend cannot fit a free `tau_party` on
three parties (note 0029 §D29.3), so it is fixed at a value the house is willing to
defend. `three_level` is the sampler-free REML route that would estimate both
components — and on four records it estimates both at zero, which is not evidence that
there is no party level; it is four records. That is exactly why the house fixes the
spread between parties at two points rather than at the zero four records cannot rule
out.

In [ ]:
from axiom.meta import Pooled, PoolPriors, PoolSpec, ThreeLevel, pool, three_level, variance_shares

fits = {}
for key, priors in (("study", PoolPriors(mu_scale=0.3, tau_scale=0.1)),
                    ("nested", PoolPriors(mu_scale=0.3, tau_scale=0.1, tau_party_fixed=0.02))):
    out = pool(PoolSpec(family="search", effect_key=key, priors=priors, mass=0.9), poolable,
               backend="laplace", draws=4000, seed=0, estimands=pool_estimands)
    assert isinstance(out, Pooled), out
    fits[key] = out.result
y = [r.estimate for r in poolable.records]; se = [r.se for r in poolable.records]
party_index = [list(H.PARTIES).index(r.contributor) for r in poolable.records]
reml: ThreeLevel = three_level(y, se, list(range(len(y))), party_index, mass=0.9)

table([["study", f"{fits['study'].n_effects} effects", f"{fits['study'].mu.mean:+.4f}", fits["study"].mu.interval.text(),
        f"{fits['study'].mu.interval.width:.4f}", f"{fits['study'].tau.mean:.4f}", "—"],
       ["nested (tau_party fixed 0.02)", f"{fits['nested'].n_effects} effects / {fits['nested'].n_parties} parties",
        f"{fits['nested'].mu.mean:+.4f}", fits["nested"].mu.interval.text(), f"{fits['nested'].mu.interval.width:.4f}",
        f"{fits['nested'].tau.mean:.4f}", f"{fits['nested'].tau_party.mean:.4f}"],
       ["three-level REML", f"{reml.k} records / {reml.n_parties} parties", f"{reml.estimate:+.4f}", reml.interval.text(),
        f"{reml.interval.width:.4f}", f"{reml.tau_study:.4f}", f"{reml.tau_party:.4f}"]],
      headers=("pool", "levels", "mu", "90% interval on mu", "width", "tau (within)", "tau_party"),
      title="the share of conversions search drives, three beliefs about the same four records")
print("commensurability, as recorded on the pool:", fits["nested"].detail["commensurability"][:120], "…")
shares = reml.shares(float(np.mean(np.asarray(se) ** 2)))
print(f"variance shares by REML (sampling, within party, between parties): {shares[0]:.0%}, {shares[1]:.0%}, {shares[2]:.0%}")

In [ ]:
nested = fits["nested"]
table([[s_.effect, s_.party, f"{s_.estimate:+.3f}", f"{s_.theta:+.3f}", f"{s_.toward:+.3f}", f"{s_.analytic:.2f}"] for s_ in nested.shrinkage],
      headers=("study", "party", "raw", "shrunk", "toward", "shrinkage"), title="each study pulled toward its own party")
moved = max(nested.shrinkage, key=lambda s_: abs(s_.theta - s_.estimate))
order = sorted(range(len(nested.shrinkage)), key=lambda i: nested.shrinkage[i].estimate)
fig = dumbbell([f"{nested.shrinkage[i].effect}  ({nested.shrinkage[i].party})" for i in order],
               [nested.shrinkage[i].estimate for i in order], [nested.shrinkage[i].theta for i in order],
               before_label="what the experiment reported", after_label="what the pool believes",
               title="Each study pulled toward its own party, not toward the family",
               subtitle=f"family mean {nested.mu.mean:+.3f}; under the nested tree acme's two tests shrink toward each other first",
               x_title="relative lift of search")
mark_x(fig, nested.mu.mean, text="family mean")
caption(fig, f"{moved.effect} moves most, by {abs(moved.theta - moved.estimate):.3f}. AC-04 has the smallest standard error in the "
             "corpus and the largest estimate, and the pool does not know why — notebook 3 does. The party level shrinks it "
             "toward AC-02 before it shrinks it toward anyone else.")

## 4 · A prior that knows whose it is

The pool becomes the next study's prior, and there are two different next studies.
Another search test for northwind crosses one level; a search test for a fourth
advertiser — one the house has never worked for — crosses both, and deserves a prior
wider by the spread between parties.

In [ ]:
from axiom.meta import prior_from_pool

rows = []
for target, party in (("mu", ""), ("predictive", ""), ("same_party", "northwind"), ("new_party", "")):
    prior, line = prior_from_pool(nested, target=target, party=party)
    rows.append([target, party or "—", f"{prior.hyper['mu']:+.4f}", f"{prior.hyper['sigma']:.4f}", line.kind])
table(rows, headers=("target", "party", "prior mean", "prior sd", "ledger kind"), title="four priors from one pool")
fourth, crossing = prior_from_pool(nested, target="new_party")
print(crossing.statement)

The fourth advertiser has a scope of its own, and the pool does not live in it. The
only way an artifact reaches a second scope is `Catalog.transfer`, and the `LedgerLine`
is not optional: the line is stored in the target scope as an artifact in its own right,
and `crossings()` is the list somebody can find six months later.

In [ ]:
from axiom.core import LedgerLine
from axiom.io import CrossScopeError, Program

initech = Program(party="initech", program="incrementality", started="2026-11-30", description="a fourth advertiser; first search test in Q1")
catalog.register(initech)
pool_digest = builders["northwind"].store.put(nested, label="search pool 2026", tags={"family": "search", "quantity": "relative_change"})
try:
    catalog.store(initech).get(pool_digest)
except CrossScopeError as err:
    print("initech asks for the pool before any transfer ->", type(err).__name__)

licence = LedgerLine(kind="scope_crossing",
                     statement="the 2026 search pool, three parties, is handed to initech as the predictive prior for a new party under the house's aggregated-benchmark terms",
                     detail={"target": "new_party", "prior_mu": f"{fourth.hyper['mu']:.4f}", "prior_sigma": f"{fourth.hyper['sigma']:.4f}"})
line_digest = catalog.transfer(pool_digest, source=programs["northwind"], target=initech, line=licence)
print("initech can now read it:", catalog.store(initech).get(pool_digest).mu.mean == nested.mu.mean)
table([[e.scope, e.label, e.derived_from[:12] + "…"] for e in catalog.crossings()],
      headers=("into scope", "label", "licensed by"), title="every artifact that crossed a boundary this year")
print("the licence itself:", catalog.store(initech).get(line_digest).statement[:100], "…")

## 5 · The model, calibrated

Back to where notebook 1 started. Northwind's media-mix model said search contributed
a number that was 1.4 times the truth, because demand drove both spend and
conversions. NW-01 measured the lift search delivers per market-week in eight weeks of
randomized holdout. `agreement` asks whether the model, realizing the same estimand on
its own posterior, agrees with the experiment.

In [ ]:
from axiom.calibrate import agreement
from axiom.surface import fit

w0 = last_year["northwind"]
panel = w0.panel()
spec = H.analyst_spec(panel, w0.party)
plain = fit(spec, panel, backend="laplace", draws=600, seed=0)
nw01 = reads["NW-01"].measurement
before = agreement(plain, nw01, seed=0)
print(f"the experiment says {nw01.estimate:+.2f} ± {nw01.se:.2f}; the model realizes {before.posterior_mean:+.2f} ± {before.posterior_sd:.2f}")
print(f"agreement: {before.verdict} (z = {before.z:+.2f}, inside the model's 90% interval: {before.inside})")

Two routes fold the experiment in. The **prior route** (`derive_prior`) turns the
measurement into a prior on the search amplitude — and with two thousand market-weeks
of confounded data behind the likelihood, an amplitude prior is overruled. The
**likelihood route** (`fit_calibrated`) adds the measurement as a term in the
likelihood on the *realized estimand*, so it pulls every parameter the lift depends on,
the curve's shape included. On a surface with carryover the constraint needs the dose
path the experiment acted on.

In [ ]:
from axiom.adapters import contribution
from axiom.calibrate import derive_prior, fit_calibrated
from axiom.estimands import realize

est = nw01.estimand
beta = plain.posterior.flat("beta_search")
draws = realize(est, plain, assume_identified=True, keep_draws=True, seed=0)
prior_route = derive_prior([nw01], spec, "search", beta_draws=beta, contribution_draws=np.asarray(draws.draws), family="lognormal")
via_prior = fit(prior_route.spec, panel, backend="laplace", draws=600, seed=0)
via_likelihood = fit_calibrated(spec, panel, [nw01], backend="laplace", draws=600, seed=0,
                                doses={"search": w0.doses["search"].mean(axis=0)})
truth_total = float(w0.contribution("search").sum())
rows = []
for label, result in (("uncalibrated", plain), ("prior route", via_prior), ("likelihood route", via_likelihood)):
    c = contribution(result, "search", seed=0)
    a = agreement(result, nw01, seed=0)
    rows.append([label, f"{result.posterior.flat('beta_search').mean():.1f}", f"{c.summary.mean:,.0f}", f"{c.summary.mean / truth_total:.2f}x",
                 f"{a.posterior_mean:+.2f}", a.verdict, f"{a.z:+.2f}"])
table(rows, headers=("fit", "beta_search", "contribution", "vs truth", "realized lift / market-week", "agreement", "z"),
      title=f"northwind search: the experiment says {nw01.estimate:+.2f} ± {nw01.se:.2f}; the truth is {truth_total:,.0f} conversions")
print("constraint on the record:", via_likelihood.provenance["constraints"][0]["name"], "| observed", via_likelihood.provenance["constraints"][0]["observed"])

In [ ]:
from axiom.surface import response_band

grid = list(np.linspace(0.0, float(w0.doses["search"].max()) * 0.8, 25))
bands = {label: response_band(result, "search", doses=grid, mass=0.9, seed=0)
         for label, result in (("uncalibrated", plain), ("likelihood route", via_likelihood))}
truth_curve = [float(w0.mean({**w0.doses, "search": np.full_like(w0.doses["search"], d)}).mean()) for d in grid]
fig = curve_band(grid, bands["uncalibrated"].mean, bands["uncalibrated"].lower, bands["uncalibrated"].upper,
                 label="uncalibrated", color=ORANGE, title="The search response, before and after one experiment",
                 subtitle="expected conversions per market-week against weekly search spend, with 90% bands; the world's truth dotted",
                 x_title="search spend (kUSD per market-week)", y_title="conversions per market-week")
curve_band(grid, bands["likelihood route"].mean, bands["likelihood route"].lower, bands["likelihood route"].upper,
           label="likelihood route", color=BLUE, fig=fig)
fig.add_scatter(x=grid, y=truth_curve, mode="lines", line={"color": MUTED, "width": 2, "dash": "dot"}, name="the truth")
caption(fig, "One eight-week holdout on a quarter of the map moves the curve part of the way toward the truth and no further: "
             "the constraint is one number with a standard error against two thousand rows that all say the same wrong thing. "
             "The agreement check still reports tension, which is the honest state to leave the model in.")

## 6 · What the house can now say, and what it cannot

To northwind: search delivers a lift the model overstated; the calibrated model still
sits above the experiment, and the next search test's prior — `same_party` — is on
the table above. To acme: two search tests agree with each other and one of them is
over the markets that reported. To globex: the search holdout reached seven markets
in ten, and the complier effect is on the ledger, not in the pool. To initech: a prior
from three parties, licensed across the boundary, with the line that says so.

What it cannot say is the sentence it started the year wanting to say — "search drives
seven percent" — as a fact about search. It is a family mean over three parties with a
within-party spread and a between-party spread, and the record of how it was made is
the product.

## What notebook 5 established

1. **Pooling is not transferring.** A complier effect is refused, a population
   difference is downgraded and recorded, and the definition of the outcome is checked
   against each party's registry — today, and as it would have been in March.
2. **Four records from three parties are not four draws.** The nested pool widens the
   family mean's interval by a fifth against the flat pool and shrinks each of acme's
   tests toward the other first; REML on four records finds no party level, which is
   the reason to fix one rather than the reason to omit it.
3. **A prior knows whose it is.** Four targets from one pool, and the widest is the one
   a new party deserves.
4. **Crossing a scope is a transfer with a ledger line**, findable from either end.
5. **One experiment moves a confounded model part of the way**, through the likelihood
   rather than the prior, and the agreement check says how far is left.